In [1]:
import pandas as pd
import numpy as np

Chargement data

In [2]:
JSON_IMAGES = "/workspace/data/raw/ImageAndXML_Data_Images.json"
JSON_GT = "/workspace/data/raw/ImageAndXML_Data_GT.json"
JSON_OCR = "/workspace/data/raw/ImageAndXML_Data_OCR.json"
df_images = pd.read_json(JSON_IMAGES)

# Chargement des GT -> DataFrame df_gt
df_gt = pd.read_json(JSON_GT)

# Chargement des OCR-> DataFrame df_ocr
df_ocr = pd.read_json(JSON_OCR)

In [3]:
#Preview
display(df_images.head(3))
display(df_gt.head(3))
display(df_ocr.head(3))

,image_filename,image_type,image_base64
0,0000223278.tif,tif,SUkqAHL9AACB/4H/gf+B/4H/h/+B/4H/gf+B/4H/h/+B/4...
1,0001136521.tif,tif,SUkqAAZJAQCBAIEAgQCBANoAAN6f/wAr7QAA3qf/gQCBAI...
2,0001139626.tif,tif,SUkqAKRsAQCBAIEA9QABdEGBAMUAAQtjgQD+AAExRKkAAB...


,region_id,entity,points,image_filename,image_width,image_height
0,page1_pos1,other,"651.955,733.089 678.475,733.089 678.475,848.01...",0000223278.tif,762,1000
1,page1_pos2,other,"134.214,707.833 222.609,707.833 222.609,798.76...",0000223278.tif,762,1000
2,page1_pos3,other,"505.472,603.013 606.494,603.013 606.494,662.36...",0000223278.tif,762,1000


,region_id,line_id,word_id,text,confidence,points,image_filename,image_width,image_height
0,Page1_Block1,Page1_Block1_l1,Page1_Block1_l1_w1,r>,0.165,"247,26 275,26 275,49 247,49",0000223278.tif,762,1000
1,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w1,B,0.430,"158,77 171,77 171,90 158,90",0000223278.tif,762,1000
2,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w2,AT,0.660,"175,77 210,77 210,91 175,91",0000223278.tif,762,1000


Graphe

In [4]:
def classify_conf(c):
    if c < 0.5:
        return "Faible"
    elif c < 0.8:
        return "Moyenne"
    return "Forte"

df_ocr["confidence_level"] = df_ocr["confidence"].apply(classify_conf)

In [5]:
import plotly.graph_objects as go
import ipywidgets as widgets
import math

page_size = 20
images = sorted(df_ocr["image_filename"].unique())
num_pages = math.ceil(len(images) / page_size)

def get_page_df(page):
    start = page * page_size
    end = start + page_size
    imgs = images[start:end]
    return df_ocr[df_ocr["image_filename"].isin(imgs)]

def plot_page(page, seuil):
    df_page = get_page_df(page)

    fig = go.Figure()

    # 1) Boxplot sans points
    fig.add_trace(go.Box(
        x=df_page["image_filename"],
        y=df_page["confidence"],
        boxpoints=False,
        name="Confidence",
        marker_color="lightgray"
    ))

    # 2) Détection des outliers selon seuil dynamique
    outliers = df_page[df_page["confidence"] < seuil]

    # 3) Points non-outliers (gris)
    non_outliers = df_page[df_page["confidence"] >= seuil]

    fig.add_trace(go.Scatter(
        x=non_outliers["image_filename"],
        y=non_outliers["confidence"],
        mode="markers",
        marker=dict(size=6, color="lightgray"),
        hoverinfo="skip",
        name="Non-outliers"
    ))

    # 4) Outliers colorés selon confidence
    fig.add_trace(go.Scatter(
        x=outliers["image_filename"],
        y=outliers["confidence"],
        mode="markers",
        marker=dict(
            size=9,
            color=outliers["confidence"],
            colorscale="RdYlGn",
            showscale=True
        ),
        text=[
            f"Image: {row['image_filename']}<br>"
            f"Confidence: {row['confidence']:.3f}<br>"
            f"Texte: {row['text']}<br>"
            f"Word ID: {row['word_id']}"
            for _, row in outliers.iterrows()
        ],
        hovertemplate="%{text}<extra></extra>",
        name="Outliers"
    ))

    fig.update_layout(
        title=f"Confidence OCR — page {page+1}/{num_pages} — seuil Confidence={seuil}",
        xaxis_title="Image",
        yaxis_title="Confidence",
        xaxis_tickangle=45,
        height=700,
        width=1200
    )

    fig.show()

# Sliders
slider_page = widgets.IntSlider(
    value=0,
    min=0,
    max=num_pages - 1,
    step=1,
    description="Page"
)

slider_seuil = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description="Seuil confidence"
)

widgets.interact(plot_page, page=slider_page, seuil=slider_seuil)


interactive(children=(IntSlider(value=0, description='Page', max=25), FloatSlider(value=0.5, description='Seui…

<function __main__.plot_page(page, seuil)>